# Q5 ML Signal Generation Framework
###   *by Dr. W. Vera-Tudela*

##### 1. Introduction & Objective

This is a custom backtesting framework built from scratch in Jupyter Notebook with supporting functions in a separate Python file. The framework uses machine learning and walk-forward validation to predict the price trend in the next 5 days as a trading strategy to either buy, hold, or sell. It uses three different ML models, compares them agains a simple MA signal generator (Q1).  

The aim is not to predict stock prices. But rather to identify recurring patterns in price, volume, and cross-asset data that have historically preceded positive or negative returns, and use those patterns to generate better-timed entry and exit signals than simple moving average crossovers.  

We're not just building an ML model — we're testing a hypothesis: volume-confirmed signals outperform price-only signals. That's testable, falsifiable, and genuinely interesting.

Objectives:
- Provide an evidence-based decision-making tool: Test your trading hypotheses with historical data.
- Allow strategy refinement: ML, MA based signal, or "buy and hold".
- Three different ML models: Logistic Regression, Random Forest Classifier, and XG Boost Classifier.
- Performance comparison: Benchmark the ML strategy against a simpler model.
- Flexibility: Custom-built framework allows for complete control over testing parameters

This framework provides a solid foundation for quantitative trading strategy development and evaluation.

### This framework builds on Q1, but instead of using a simple MA-based signal, it implements Machine Learning using up to 15 price and volume based fatures.  


## 2. Data Pipeline

The structure of the data flow is as follows:

The pipeline:

1. Data Download
   - Download adjusted prices and volumes for all assets
     
2. Feature Engineering
   - 15 features with clear economic rationale in total
   - 7 price-based
   - 5 volume-based
   - 3 cross-asset
        
3. Target Variable
   - 5-day forward return, binarized, non-overlapping
     
4. Walk-Forward Validation
   - Split into train and testing sets
   - Non overlaping
   - Expanding window

5. Model Training
   - Logistic Regression
   - Random Forest Classifier
   - XGBoost Classifier

6. Signal Generation on test window
   - Buy & Sell indicators
   - Hold otherwise

7. Signal Evaluation Metrics
   - Accuracy
   - Precision
   - Recall
   - Confusion matrix comparison

8. Backtesting
   -  Feeding ML signal into Q1 backtesting engine
   -  Comparison of ML predictions vs MA crossover baseline from Q1
  
9. Feature Importance
   - Which features drove predictions
   - Most relevant ones
   - Which fall below the baseline

10. Volume Hypothesis Test — does volume add predictive power beyond price alone?


11. Visualisation
    - Displays equity curves to track portfolio growth
    - Comparison of end portfolio value

This consolidated framework provides a complete pipeline from data acquisition through strategy testing to performance evaluation and visualization.  

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

import sys
sys.path.append('../utils')   # path relative to the notebook
from Q5_functions import engineer_features, model_training, compute_metrics_ml, signal_backtest
from common import fetch_portfolio_data

In [2]:
# Load data
starting_capital = 10_000
period = 10
end = pd.Timestamp.today(tz="UTC").normalize()
start = end - pd.DateOffset(years=period)

tickers = ['AAPL', 'BTC-USD', 'SPY', 'GLD']

prices, volumes = fetch_portfolio_data(tickers, start, end)

## 3. Feature engineering

A total of 15 features with clear economic rationale have been selected as follows:

### Price-based (7):  
- 5-day, 20-day, 60-day momentum (log return over window)
- MA20/MA50 ratio (are we in a golden cross regime?)
- Stochastic Oscillator, which measures price position within the recent high-low range.
- Bollinger Band position (where is the price within its recent range?)
- Realized volatility — 20-day rolling std of daily returns

### Volume-based (5):  
- Volume ratio — today's volume / 20-day average volume
- OBV normalized — On-Balance Volume, normalized to remove scale
- Price-volume divergence — 5-day price change / 5-day volume change
- Volume trend — 5-day MA of volume / 20-day MA of volume
- High-volume day flag — binary, 1 if volume > 2x average

### Cross-asset (3):  
- BTC/AAPL rolling 20-day correlation (regime indicator)
- Gold return over the past 5 days (risk-off signal)
- SPY 5-day momentum (broad market context)

This is done by the function `engineer_features`.

In [3]:
TARGET = 'AAPL'
features = engineer_features(prices, volumes, TARGET)

# 5-day forward log return
forward_return = np.log(prices[TARGET].shift(-5) / prices[TARGET])

# Binary target: 1 if positive, 0 if negative
target = (forward_return > 0).astype(int)

df = features.copy()
df['target'] = target
df = df.dropna()

# Non-overlapping 5-day samples
df_sampled = df.iloc[::5].copy()
print(df_sampled.shape)
print(df_sampled['target'].value_counts(normalize=True))

(719, 18)
target
1    0.584145
0    0.415855
Name: proportion, dtype: float64


## 4. Model training

Three different models have been selected and are trained for comparison:
- Logistic Regression
- Random Forest Classifier
- XGBoost Classifier

We also have the option to select different features by type:
- Price only
- Price and volume
- All features (price, volume, and cross-asset)

This is done by the function `model_training`.

In [4]:
feature_cols = [c for c in df_sampled.columns if c != 'target']

price_only_cols = ['mom_5', 'mom_20', 'mom_60', 'MA_ratio', 
                   'Stoch_14', 'BBP', 'real_vola_20', 'positive_days']

price_volu_cols = ['mom_5', 'mom_20', 'mom_60', 'MA_ratio', 'Stoch_14', 'BBP', 'real_vola_20', 'positive_days',
                   'VR', 'OBV_normalized', 'pv_divergence', 'volu_trend', 'high_volu_flag']

results_df = model_training(price_volu_cols, df_sampled)g

SyntaxError: invalid syntax (1883085465.py, line 9)

## 5. Compute metrics

After the models have been trained, the different metrics are calculated:

- Accuracy: How often the model's prediction matches the actual outcome (true positives and true negatives).
- Precision: Of all days predicted as 'up', what fraction actually were up?
- Recall: Of all actual 'up' days, what fraction did the model correctly identify?

In addition, the confusion matrices are shown to visualize these results for each model.

This is done by the function `compute_metrics`.

In [ ]:
compute_metrics_ml(df_sampled,results_df)

## 6. Feature importance

Each feature is then ranked by importance from the XGB classifier and compared again the baseline to see which ones are the most relevant for the models.  
These are plotted in a bar chart for ease of visualization.

In [ ]:
# Get feature importance from last trained XGBoost model
# Better: retrain once on full dataset for importance analysis
xgb_full = XGBClassifier(n_estimators=100, random_state=42, 
                          verbosity=0, eval_metric='logloss')
xgb_full.fit(
    StandardScaler().fit_transform(df_sampled[feature_cols]),
    df_sampled['target']
)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_full.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.axvline(x=1/len(feature_cols), color='r', linestyle='--', 
            label='Equal importance baseline')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Signal backtesting

Once the predictions have been calculated, these are fed into the backtester from Q1 to calculate the portfolio value over time.  
In addition, the portfolio value is also calculated using the simple methodology from Q1 where the buy/sell signals are generated from the 20- and 50-day rolling averages.

This is done by the function `signal_backtest`.

In [ ]:
# Map walk-forward predictions back to daily dates
# results_df has dates from non-overlapping 5-day samples
# Need to forward-fill signal to daily frequency

signal_ML = pd.Series(
    results_df['pred_xgb'].values,
    index=results_df['date']
).reindex(df.index).ffill().fillna(0)

# Now signal_ml is 1 (long) or 0 (flat) on every day
# Feed into Q1 backtester with AAPL prices

In [ ]:
dataA = pd.DataFrame(index=prices.index)

dataA['Close'] = prices[TARGET]
dataA.dropna()

# Calculate 20-day / 50-day rolling averages and generate signals based on crossovers
MA20 = dataA['Close'].rolling(window=20).mean() # Q1
MA50 = dataA['Close'].rolling(window=50).mean() # Q1
signal_Q1 = (MA20 > MA50).astype(int) # Q1

dataQ1 = signal_backtest(dataA, signal_Q1, starting_capital)
dataML = signal_backtest(dataA, signal_ML, starting_capital)

print(f"Q1:  ${dataQ1.iloc[-1]['Total']:,.0f}")
print(f"ML:  ${dataML.iloc[-1]['Total']:,.0f}")
print(f"HODL:  ${dataML.iloc[-1]['Buy_Hold']:,.0f}")

## 8. Results visualization

Finally a comparison plot is show to benchmark the new ML strategy against the simpler Q1 model and the option of "buy and hold".

In [ ]:
plt.figure(figsize=(16, 9))
colors = sns.color_palette("colorblind")

plt.plot(dataQ1['Total'], color=colors[0], label = 'Q1')
plt.plot(dataML['Total'], color=colors[1], label = 'ML')
plt.plot(dataML['Buy_Hold'], color=colors[2], label = 'HODL')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 9. Conclusions

- The core finding is clear: Volume-confirmed features improve ML signal quality over price-only features, but neither approach beats simple momentum rules on a trending asset over a bull market period.

- ML signal generation with volume-confirmed features outperforms price-only ML by 36% in backtested returns, confirming that volume carries genuine predictive information beyond price alone. However, both ML approaches underperform the simple MA crossover baseline from Q1, suggesting that for trending assets in sustained bull markets, momentum-following rules remain more effective than supervised classification. The ML approach may show greater relative advantage in mean-reverting or range-bound market conditions where momentum signals fail — a hypothesis left for future testing.

- Volume beats cross-asset. The local information (how much conviction is behind this price move) is more predictive than the global context (what other assets are doing).

- Volume Ratio (VR) was the single most important feature in the price+volume model, ranking above all price-based features. This directly confirms the hypothesis that volume carries predictive information beyond price alone. The high-volume flag (binary) added no value beyond the continuous ratio — magnitude matters more than threshold crossing.

## 10. Limitations & Next Steps

The three most important limitations of this model are the following:  

1. The results may present survivor bias as it has been trained using a single bullish asset.
2. The same features have been used for all models, even though the results varied. Different features could be selected per model to maximize output.
3. A fixed 5-day forward window has been used as a predictor. Different windows could be used to see if the model performs differently.

Some interesting next steps to go deeper on this project would be:
- Remove features rated below the baseline, as fewer, higher-quality features with the same training data usually improve generalization.
- Test across a broader universe, including historical losers.
- Test the ML approach in mean-reverting or range-bound market conditions where momentum signals fail.

Just to name a few.